In [3]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../..")

1. Install Required Libraries
pip install -r requirements.txt

2. Azure Blob Setup

In [5]:
from azure.storage.blob import BlobServiceClient
from src.config import AZURE_CONNECTION_STRING, CONTAINER_DARWIN_REALTIME

blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_DARWIN_REALTIME)

3. Import Core Libraries

In [6]:
from confluent_kafka import Consumer
import json
import pandas as pd
from lxml import etree
from datetime import datetime
from io import StringIO

4. Azure Upload Function

In [7]:
def upload_to_azure(records, blob_name):
    df = pd.DataFrame(records)

    buffer = StringIO()
    df.to_csv(buffer, index=False)

    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(buffer.getvalue(), overwrite=True)

    print(f"Uploaded {len(records)} records → {blob_name}")

5. Batch + File Naming Logic

In [8]:
BATCH_SIZE = 100
UPLOAD_SIZE = 2000

def generate_blob_name():
    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    return f"raildata_{timestamp}.csv"

6. Config and Main Pipleine

In [10]:
from src.config import REAL_TIME_FEEDS_CONFIG, REAL_TIME_FEEDS_TOPIC
consumer = Consumer(REAL_TIME_FEEDS_CONFIG)
consumer.subscribe([REAL_TIME_FEEDS_TOPIC])

print("Streaming Darwin data → Azure Blob...")

buffer = []
chunk_counter = 0
current_blob = generate_blob_name()

try:
    while True:
        msg = consumer.poll(1.0)

        if msg is None:
            continue

        if msg.error():
            print("Error:", msg.error())
            continue

        try:
            # Decode Kafka message
            raw_str = msg.value().decode('utf-8')
            data = json.loads(raw_str)

            # Extract XML
            if "bytes" not in data:
                continue

            xml_data = data["bytes"]
            root = etree.fromstring(xml_data.encode('utf-8'))

            # Skip heartbeat messages
            if root.find(".//{*}TS") is None:
                continue

            # Extract train data
            for ts in root.findall(".//{*}TS"):
                rid = ts.attrib.get("rid")
                uid = ts.attrib.get("uid")
                ssd = ts.attrib.get("ssd")

                for loc in ts.findall(".//{*}Location"):
                    record = {
                        "rid": rid,
                        "train_id": uid,
                        "date": ssd,
                        "station": loc.attrib.get("tpl"),
                        "pta": loc.attrib.get("pta"),
                        "ptd": loc.attrib.get("ptd"),
                        "wta": loc.attrib.get("wta"),
                        "wtd": loc.attrib.get("wtd"),
                        "ingestion_time": datetime.utcnow(),
                        "source_file": current_blob   # traceability
                    }

                    buffer.append(record)

            # Batch upload
            if len(buffer) >= BATCH_SIZE:
                upload_to_azure(buffer, current_blob)

                chunk_counter += len(buffer)
                buffer.clear()

                # File rotation
                if chunk_counter >= UPLOAD_SIZE:
                    print("Chunk limit reached → rotating file")
                    current_blob = generate_blob_name()
                    chunk_counter = 0

        except Exception as e:
            print("Processing error:", e)
            continue

except KeyboardInterrupt:
    print("Stopping stream...")

finally:
    if buffer:
        upload_to_azure(buffer, current_blob)

    consumer.close()
    print("Consumer closed")

Streaming Darwin data → Azure Blob...
Uploaded 100 records → raildata_20260426_151549.csv
Uploaded 106 records → raildata_20260426_151549.csv
Uploaded 110 records → raildata_20260426_151549.csv
Uploaded 112 records → raildata_20260426_151549.csv
Uploaded 100 records → raildata_20260426_151549.csv
Uploaded 108 records → raildata_20260426_151549.csv
Uploaded 115 records → raildata_20260426_151549.csv
Uploaded 133 records → raildata_20260426_151549.csv
Uploaded 100 records → raildata_20260426_151549.csv
Uploaded 103 records → raildata_20260426_151549.csv
Uploaded 112 records → raildata_20260426_151549.csv
Uploaded 101 records → raildata_20260426_151549.csv
Uploaded 101 records → raildata_20260426_151549.csv
Uploaded 114 records → raildata_20260426_151549.csv
Uploaded 102 records → raildata_20260426_151549.csv
Uploaded 100 records → raildata_20260426_151549.csv
Uploaded 128 records → raildata_20260426_151549.csv
Uploaded 102 records → raildata_20260426_151549.csv
Uploaded 103 records → rai